In [20]:
import torch
from d2l import torch as d2l

In [21]:
# Input & label Sequence 생성

# corpus 길이가 T, Sequence 길이가 n이면:
# - array.shape: [T - n, n + 1]
# - X.shape = Y.shape = [T - n, n]

def time_machine_init(
    self: d2l.TimeMachine,
    batch_size: int,
    num_steps: int,
    num_train: int = 10000,
    num_val: int = 5000,
) -> None:
    
    super(
        d2l.TimeMachine,
        self,
    ).__init__()
    
    self.save_hyperparameters()
    
    corpus, self.vocab = self.build(
        self._download()
    )
    
    subsequences: list[list[int]] = [
        corpus[
            start:start + num_steps + 1
        ]
        for start in range(
            len(corpus) - num_steps
        )
    ]
    
    # [T-n, n+1]
    array = torch.tensor(
        subsequences,
        dtype=torch.long,
    )
    
    # Input: [T-n, n] 
    # 마지막 Token 제외
    self.X = array[
        :,
        :-1,
    ]
    
    # Target: [T-n, n]  
    # 첫 번째 토큰 제외
    self.Y = array[
        :,
        1:,
    ]
    
d2l.TimeMachine.__init__ = (
    time_machine_init
)

In [22]:
# Train & Validation DataLoader

def time_machine_get_dataloader(
    self: d2l.TimeMachine,
    train: bool,
) -> torch.utils.data.DataLoader:
    
    indices = (
        slice(
            0,
            self.num_train,
        )
        if train
        else slice(
            self.num_train,
            self.num_train + self.num_val,
        )
    )
    
    return self.get_tensorloader(
        (
            self.X,
            self.Y,
        ),
        train,
        indices,
    )
    

d2l.TimeMachine.get_dataloader = (
    time_machine_get_dataloader
)

In [23]:
# Minibatch의 한 칸 Shift 관계 Check

data = d2l.TimeMachine(
    batch_size=2,
    num_steps=10,
)

X, Y = next(
    iter(
        data.train_dataloader()
    )
)

print(
    "X:"
)
print(X)

print(
    "\nY:"
)
print(Y)

print(
    "\nX shape:",
    tuple(X.shape),
)

print(
    "Y shape:",
    tuple(Y.shape),
)

print(
    "Shift relation:",
    torch.equal(
        X[:, 1:],
        Y[:, :-1],
    ),
)

X:
tensor([[21, 21, 19,  2,  4, 21,  0,  2, 21, 21],
        [ 0, 21,  9,  6,  0, 23,  6, 19, 26,  0]])

Y:
tensor([[21, 19,  2,  4, 21,  0,  2, 21, 21,  6],
        [21,  9,  6,  0, 23,  6, 19, 26,  0, 26]])

X shape: (2, 10)
Y shape: (2, 10)
Shift relation: True
